# GPU 중단 복구 — 남은 조건 실행
GPU 할당 후 아래 셀을 순서대로 실행합니다. 기존 전체 실행 셀을 실행하지 않습니다. BIM/PGD 부모 결과를 검사하고 PGD 11조건을 상속합니다. 환경 불일치 시 중단합니다. 이 노트북은 첫 복구 시도용입니다. 재중단되면 저장 산출물을 다시 감사해야 하며 전체 재실행하지 마세요. Drive 백업에는 비공개 가중치가 포함됩니다. 폴더를 공개하지 마세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys,os,json,zipfile,shutil,datetime,hashlib
ARCHIVE=Path('/content/drive/MyDrive/AdversarialAI_GPU_handoff_20261002.zip')
WORK=Path('/content/adversarial_full');WORK.mkdir(exist_ok=True)
REPO=WORK/'repo'
import re
COMMIT=input('Reviewed execution SHA (40 hex): ').strip()
if not re.fullmatch(r'[0-9a-f]{40}',COMMIT): raise ValueError('Explicit reviewed SHA required')
print('Pinned recovery source:',COMMIT)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/heechan9/AdversarialAI_Security.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==COMMIT
if not ARCHIVE.is_file():
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    sys.path.insert(0,str(REPO/'verification'))
    from colab_inputs import find_archive,download_archive
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    ARCHIVE=download_archive(service,find_archive(service),WORK/'handoff.zip')
with zipfile.ZipFile(ARCHIVE) as z:
    selected=[i for i in z.infolist() if i.filename.startswith(('data/train-prepared/','data/valid/','data/test/','models/'))]
    for i in selected:
        target=(WORK/i.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'unsafe archive path'
        assert (i.external_attr >> 16) & 0o170000 != 0o120000, 'symlink in archive'
        z.extract(i,WORK)
for p in (WORK/'models').glob('*.h5'):
    (REPO/'models').mkdir(exist_ok=True);shutil.copy2(p,REPO/'models'/p.name)
print('입력 압축 해제 완료. 학습 시작 전 코드가 이미지와 모델 해시를 검사합니다.')


In [ ]:
assert 'WORK' in globals() and 'REPO' in globals(), '첫 번째 셀을 먼저 오류 없이 완료하세요.'
VENV=WORK/'venv'
PY=str(VENV/'bin/python')
def run_logged(command,env=None):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env) as process:
        for line in process.stdout: print(line,end='',flush=True)
        if process.wait(): raise RuntimeError('위 설치/검사 로그를 확인하세요.')
# Colab system Python may omit ensurepip; host pip can populate a pip-free venv.
if not Path(PY).is_file():run_logged([sys.executable,'-m','venv','--without-pip',str(VENV)])
run_logged([sys.executable,'-m','pip','--python',PY,'install','tensorflow[and-cuda]==2.21.0','keras==3.15.1','numpy==2.3.5','Pillow','pandas','scikit-learn','matplotlib','pyyaml'])
ENV=os.environ.copy();ENV['PYTHONPATH']=str(REPO/'src')+':'+str(REPO)
ENV['TF_NUM_INTRAOP_THREADS']='2'
ENV['TF_NUM_INTEROP_THREADS']='1'
ENV['TF_FORCE_GPU_ALLOW_GROWTH']='true'
ENV['MPLBACKEND']='Agg'
CUDA_LIBS=sorted(str(p) for p in VENV.glob('lib/python*/site-packages/nvidia/*/lib'))
ENV['LD_LIBRARY_PATH']=':'.join(CUDA_LIBS+[ENV.get('LD_LIBRARY_PATH','')])
run_logged([PY,'-c',"import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g, 'GPU 미할당: CPU로 자동 진행하지 않습니다.'"],env=ENV)
DEST=Path('/content/drive/MyDrive/AdversarialAI_GPU_results');DEST.mkdir(exist_ok=True)
print('GPU 확인 완료. 다음 셀은 부모 결과를 감사합니다.')


In [ ]:
os.chdir(REPO)
sys.path.insert(0,str(REPO));sys.path.insert(0,str(REPO/'src'))
from verification.iterative_result_audit import audit
PARENT=DEST/'full-gpu-20261004T120332'
def locate(name):
    candidates=list(PARENT.rglob(name+'/run.json'))
    assert len(candidates)==1, ('부모 결과 경로를 유일하게 찾을 수 없음',name)
    return candidates[0].parent
BIM=locate('full-gpu-20261004T120332-bim')
PGD_PARENT=locate('full-gpu-20261004T120332-pgd')
assert audit(BIM,REPO)['result']=='FULL_OUTPUTS_CONSISTENT'
assert audit(PGD_PARENT,REPO)['checked_conditions']==11
RUN='recovery-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S')
PRIVATE=DEST/RUN
PRIVATE.mkdir(exist_ok=False)
(PRIVATE/'execution-source.json').write_text(json.dumps({'source_commit':COMMIT},indent=2))
PGD=REPO/'results/extensions/iterative'/(RUN+'-pgd')
JSMA=REPO/'results/extensions/jsma'/(RUN+'-jsma')
TRAIN=REPO/'results/extensions/adversarial_training'/(RUN+'-train')
EVAL=REPO/'results/extensions'/(RUN+'-eval')
def stage(name,out,args):
    assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==COMMIT, 'Execution source changed'
    run_logged([PY,'-m','verification.recovery_stage','--output',str(out),'--backup',str(PRIVATE/name),'--',*args],env=ENV)
print('부모 감사 완료; 복구 ID:',RUN)


In [ ]:
stage('pgd',PGD,['-m','adversarial_ai.evaluation.iterative_evaluation','--attack','pgd','--steps','20','--step-size','.005','--restarts','5','--seed','2026','--batch-size','16','--run-id',PGD.name,'--data-dir',str(WORK/'data/test'),'--resume-from',str(PGD_PARENT)])
assert audit(PGD,REPO)['result']=='FULL_OUTPUTS_CONSISTENT'


In [ ]:
stage('jsma',JSMA,['-m','adversarial_ai.evaluation.jsma_evaluation','--model','cnn','--defense','none','--theta','1','--gamma','.01','--max-steps','246','--limit','781','--run-id',JSMA.name,'--data-dir',str(WORK/'data/test')])
from verification.jsma_saved_audit import audit as audit_jsma
assert audit_jsma(JSMA,REPO)['result']=='FULL_SAVED_EVIDENCE_CONSISTENT'


In [ ]:
stage('training',TRAIN,['-m','adversarial_ai.training.adversarial_training','--model','mobilenet','--train-dir',str(WORK/'data/train-prepared/images'),'--validation-dir',str(WORK/'data/valid'),'--test-dir',str(WORK/'data/test'),'--epochs','3','--epsilon','.03','--steps','7','--step-size','.005','--seed','2026','--batch-size','8','--learning-rate','.00001','--run-id',TRAIN.name])


In [ ]:
stage('evaluation',EVAL,['verification/mobilenet_pgd7_test_evaluation.py','--trained-dir',str(TRAIN),'--test-dir',str(WORK/'data/test'),'--output-dir',str(EVAL)])
from verification.full_extension_audit import audit_all
result=audit_all(BIM,PGD,JSMA,TRAIN,EVAL,REPO)
(PRIVATE/'saved-evidence-audit.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
print('저장 증거 감사 완료. GPU 로그·lineage·공개 범위·CI 검토 및 최종 병합은 별도입니다.')
